# Module 1: Jupyter Notebook Basics & REST API
**หลักสูตร:** การพัฒนาเวิร์กโฟลว์ปัญญาประดิษฐ์สำหรับสร้างข้อความและรูปภาพบนเครื่องคอมพิวเตอร์ส่วนบุคคล  
**(Mastering Local AI Stack: Jupyter Notebook × LM Studio × ComfyUI)**

---

## 🎯 วัตถุประสงค์การเรียนรู้ (Learning Objectives)
1. เข้าใจโครงสร้างและการทำงานของ **Jupyter Notebook** (ความแตกต่างระหว่าง `Markdown Cell` และ `Code Cell`, การทำงานของ Python Kernel บน `.venv`)
2. สามารถเขียนโค้ด Python เพื่ออ่านไฟล์ข้อความภาษาไทย (`sample_article.txt`) มาประมวลผลเบื้องต้นได้
3. เข้าใจโครงสร้างข้อมูลแบบ **Dictionary / JSON** และการแปลงข้อมูลให้อยู่ในรูปแบบตารางด้วย `pandas.DataFrame`
4. สามารถใช้ไลบรารี `requests` เพื่อตรวจสอบสถานะ (Health Check) ของ Local AI Server ทั้ง **LM Studio** และ **ComfyUI** ได้

## ส่วนที่ 1: รู้จักกับ Markdown Cell และ Code Cell

ใน Jupyter Notebook พื้นที่การทำงานจะแบ่งออกเป็นกล่องๆ เรียกว่า **Cell** โดยมี 2 ประเภทหลักที่ใช้บ่อยที่สุด:
- **Markdown Cell (กล่องข้อความ):** ใช้สำหรับเขียนคำอธิบาย หัวข้อ สมการ หรือเอกสารประกอบ (เช่น กล่องที่คุณกำลังอ่านอยู่นี้)
- **Code Cell (กล่องโค้ด):** ใช้สำหรับเขียนและรันโค้ดภาษา Python โดยกดปุ่ม `Shift + Enter` เพื่อสั่งให้ **Kernel** ประมวลผลและแสดงผลลัพธ์ด้านล่างทันที

In [ ]:
# กด Shift + Enter เพื่อรัน Code Cell แรกของคุณ
import sys
import platform

print("=== ข้อมูลสภาพแวดล้อม Python (Environment Info) ===")
print(f"Operating System : {platform.system()} {platform.release()} ({platform.machine()})")
print(f"Python Version   : {sys.version.split()[0]}")
print(f"Executable Path  : {sys.executable}")

## ส่วนที่ 2: การอ่านไฟล์บทความภาษาไทย (`sample_article.txt`)

ในการทำงานกับ AI เรามักต้องดึงข้อมูลจากเอกสารภายนอกมาเป็นข้อมูลนำเข้า (Input Context) ให้กับโมเดล ในขั้นตอนนี้เราจะลองเปิดอ่านไฟล์ `../assets/sample_article.txt` ด้วยการเข้ารหัส `utf-8` เพื่อให้แสดงผลภาษาไทยได้อย่างถูกต้อง

In [ ]:
from pathlib import Path

# กำหนด Path ไปยังไฟล์บทความตัวอย่าง
article_path = Path("../assets/sample_article.txt")

if article_path.exists():
    article_text = article_path.read_text(encoding="utf-8")
    paragraphs = [p.strip() for p in article_text.split("\n\n") if p.strip()]
    
    print(f"✅ อ่านไฟล์สำเร็จ! ความยาวทั้งหมด: {len(article_text):,} ตัวอักษร")
    print(f"📌 จำนวนย่อหน้าทั้งหมด: {len(paragraphs)} ย่อหน้า\n")
    print("--- ตัวอย่างเนื้อหาย่อหน้าที่ 1 ---")
    print(paragraphs[0])
else:
    print(f"❌ ไม่พบไฟล์ที่ {article_path.resolve()}")

## ส่วนที่ 3: การจัดการข้อมูล Dictionary / JSON และ `pandas.DataFrame`

เมื่อเราสื่อสารกับ REST API ของระบบ AI (ทั้ง LM Studio และ ComfyUI) ข้อมูลที่รับ-ส่งกันจะอยู่ในรูปแบบ **JSON (JavaScript Object Notation)** ซึ่งเทียบเท่ากับโครงสร้างข้อมูลแบบ `dict` และ `list` ในภาษา Python

นอกจากนี้ เรายังสามารถนำข้อมูล JSON ที่สกัดได้มาแปลงเป็นตาราง `pandas.DataFrame` เพื่อวิเคราะห์หรือส่งออกเป็นไฟล์ CSV/Excel ได้อย่างง่ายดาย

In [ ]:
import json
import pandas as pd

# ตัวอย่างข้อมูลสเปกโมเดลที่เหมาะสมกับเครื่อง Lab (VRAM 6GB / Apple Silicon 16GB)
local_ai_models = [
    {
        "component": "LM Studio (LLM)",
        "model_name": "Qwen2.5-3B-Instruct (Q4_K_M)",
        "format": "GGUF",
        "vram_usage_gb": 2.2,
        "recommended_setting": "Context Length <= 2048-4096"
    },
    {
        "component": "LM Studio (LLM)",
        "model_name": "Llama-3.2-3B-Instruct (Q4_K_M)",
        "format": "GGUF",
        "vram_usage_gb": 2.1,
        "recommended_setting": "Context Length <= 2048-4096"
    },
    {
        "component": "ComfyUI (Diffusion)",
        "model_name": "SD 1.5 (v1-5-pruned-emaonly)",
        "format": "Safetensors",
        "vram_usage_gb": 2.0,
        "recommended_setting": "Resolution 512x512, Steps 20"
    }
]

# 1. แปลง Python List/Dict เป็นข้อความ JSON (Serialization)
json_payload = json.dumps(local_ai_models, ensure_ascii=False, indent=2)
print("=== ข้อมูลในรูปแบบ JSON String ===")
print(json_payload)

# 2. แปลงข้อมูลเป็นตาราง Pandas DataFrame
df_models = pd.DataFrame(local_ai_models)
print("\n=== ตารางสรุปการใช้ VRAM (Pandas DataFrame) ===")
display(df_models)

## ส่วนที่ 4: การเรียกใช้งาน REST API และตรวจสอบสถานะเซิร์ฟเวอร์ด้วย `requests.get()`

ก่อนที่เราจะส่งคำสั่งให้ AI ทำงานในโมดูลถัดไป เราสามารถใช้ไลบรารี `requests` ส่ง HTTP GET Request ไปตรวจสอบว่าเซิร์ฟเวอร์จำลองบนเครื่องของเรา (Localhost) เปิดทำงานอยู่หรือไม่:
- **LM Studio Local Server:** `http://localhost:1234/v1/models`
- **ComfyUI Server:** `http://127.0.0.1:8188/system_stats`

In [ ]:
import requests

def check_server_status(service_name: str, url: str) -> dict:
    """ฟังก์ชันทดสอบเชื่อมต่อ Local Server ด้วย requests.get()"""
    try:
        response = requests.get(url, timeout=3)
        if response.status_code == 200:
            return {
                "Service": service_name,
                "Endpoint": url,
                "Status": "🟢 Online (HTTP 200)",
                "Preview": str(response.json())[:90] + "..."
            }
        return {
            "Service": service_name,
            "Endpoint": url,
            "Status": f"🟡 HTTP {response.status_code}",
            "Preview": response.text[:90]
        }
    except requests.exceptions.RequestException as err:
        return {
            "Service": service_name,
            "Endpoint": url,
            "Status": "⚪ Offline (ยังไม่ได้เปิดเซิร์ฟเวอร์)",
            "Preview": type(err).__name__
        }

# ทดสอบเช็กสถานะทั้ง LM Studio และ ComfyUI
checks = [
    check_server_status("LM Studio API", "http://localhost:1234/v1/models"),
    check_server_status("ComfyUI Server", "http://127.0.0.1:8188/system_stats")
]

df_status = pd.DataFrame(checks)
display(df_status)

---
### ✅ สรุปท้ายบทเรียน Module 1
ตอนนี้คุณพร้อมแล้วสำหรับการเขียนโปรแกรมควบคุม Local AI! ใน **Module 2** ถัดไป เราจะเปิดใช้งาน **LM Studio Local Server** และใช้ไลบรารี `openai` เชื่อมต่อเพื่อสั่งให้โมเดล 3B สรุปและสกัดข้อมูลจากบทความภาษาไทยกันครับ